# Lab R.4 &mdash; Evaluation with Ragas

**About 30 minutes** &middot; Day 2 &middot; RAG, vector stores &amp; agent memory

Here you score the whole pipeline, retrieval and answer, with **Ragas**. The sandbox model is the judge. You score twice, without and with the LLM reranker from Lab R.2, and find the weakest stage.

Run the cells in order, with **Shift + Enter**. Under each cell, **You should see** says what to
expect. The shared helpers are in `rag_kit.py`, next to this notebook.

**The result:** a table of four scores for two versions of the pipeline, and which stage to fix next.

## Step 1 &mdash; Install and set up Ragas

Ragas is already in the sandbox. The first cell installs it only if it is missing. Ragas 0.4.3 imports
a Google Vertex AI class that the sandbox's newer `langchain_community` no longer has. This lab does
not use Vertex AI, so the second cell puts an empty stand-in in its place before the import.

In [ ]:
import importlib.util, site
if importlib.util.find_spec("ragas") is None:      # already in the sandbox image; install only if missing
    %pip install -q "ragas==0.4.3"
    site.addsitedir(site.getusersitepackages())    # use the new package without a kernel restart
print("ragas is installed")

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")                    # the model libraries print a lot on first import
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import rag_kit as kit

import sys, types
_stub = types.ModuleType("langchain_community.chat_models.vertexai")
_stub.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules.setdefault("langchain_community.chat_models.vertexai", _stub)

import ragas
from ragas import evaluate, EvaluationDataset, RunConfig
from ragas.metrics import (Faithfulness, ResponseRelevancy,
                           LLMContextPrecisionWithReference, LLMContextRecall)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_core.embeddings import Embeddings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

class LocalEmbeddings(Embeddings):
    """The same all-MiniLM-L6-v2 that Chroma uses: no download and no gateway call."""
    def __init__(self):
        self.fn = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [[float(x) for x in v] for v in self.fn(texts)]
    def embed_query(self, text):
        return self.embed_documents([text])[0]

judge = LangchainLLMWrapper(ChatOpenAI(model=os.environ["LAB_LLM_MODEL"], temperature=0,
                                       extra_body=kit.NO_THINK))
embeddings = LangchainEmbeddingsWrapper(LocalEmbeddings())
METRICS = [LLMContextPrecisionWithReference(), LLMContextRecall(), Faithfulness(), ResponseRelevancy()]
print("Ragas", ragas.__version__, "| judge:", os.environ["LAB_LLM_MODEL"])

**You should see:** `Ragas 0.4.3` and the name of the sandbox model.

## Step 2 &mdash; The pipeline under test

Hybrid search from Lab R.1, the LLM reranker from Lab R.2, and an **answer step**: the top 3 chunks go
into a prompt that says to answer only from them. The test set is the **6 questions** where hybrid
search missed. A good test set holds the hard cases.

In [ ]:
import re, time
chunks = kit.all_chunks()
text = {c["id"]: c["text"] for c in chunks}
col = kit.build_collection(chunks)
bm25 = kit.BM25(chunks)
questions = kit.load_questions()

def rrf(ranked_lists, c=60):                         # from Lab R.1
    scores = {}
    for ranked in ranked_lists:
        for rank, chunk_id in enumerate(ranked, start=1):
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (c + rank)
    return sorted(scores, key=lambda cid: -scores[cid])

def candidates(question, n=20):                      # hybrid search: the top 20
    return rrf([kit.vector_search(col, question, k=20), bm25.search(question, k=20)])[:n]

llm_tokens = 0

def llm_rerank(question, ids):
    """Send the 20 chunks in one prompt, numbered, and ask for the best 3."""
    global llm_tokens
    numbered = "\n".join(f"[{n}] {text[cid]}" for n, cid in enumerate(ids))
    reply, tokens = kit.chat(f"Question: {question}\n\nPassages:\n{numbered}\n\n"
                             "Which 3 passages best answer the question? "
                             "Reply with their numbers only, best first, like: 4, 0, 7", max_tokens=20)
    llm_tokens += tokens
    picked = []
    for n in map(int, re.findall(r"\d+", reply)):
        if n < len(ids) and ids[n] not in picked:     # ignore numbers that are not a position
            picked.append(ids[n])
    return picked + [cid for cid in ids if cid not in picked]

def retrieve(question, rerank=False):
    ids = candidates(question)
    ids = llm_rerank(question, ids) if rerank else ids
    return [text[cid] for cid in ids[:3]]

def answer(question, contexts):
    reply, _ = kit.chat("Answer the on-call engineer's question using ONLY the runbook passages below. "
                        "If they do not contain the answer, say so. Answer in at most 3 sentences.\n\n"
                        "Passages:\n" + "\n\n".join(contexts) + f"\n\nQuestion: {question}", max_tokens=200)
    return reply.strip()

eval_questions = [q for q in questions if q["chunk"] not in candidates(q["question"])[:3]][:6]
for q in eval_questions:
    print("-", q["question"])

**You should see:** six questions. These are the ones hybrid search got wrong in Lab R.1.

## Step 3 &mdash; Build the evaluation set

For each question Ragas needs four things: the **question**, the **chunks** retrieval returned, the
pipeline's **answer**, and a **reference** answer written by a person, from `questions.json`.

In [ ]:
def build_samples(rerank):
    samples = []
    for q in eval_questions:
        contexts = retrieve(q["question"], rerank=rerank)
        samples.append({"user_input": q["question"], "retrieved_contexts": contexts,
                        "response": answer(q["question"], contexts), "reference": q["reference"]})
    return samples

samples_hybrid = build_samples(rerank=False)
print(samples_hybrid[0]["user_input"], "\n->", samples_hybrid[0]["response"])

**You should see:** the first question and the pipeline's answer to it.

## Step 4 &mdash; Score it, twice

This scores the hybrid-only pipeline, then builds a second set **with** the reranker and scores that.
It takes a few minutes. While it runs, predict which score the reranker changes most.

- **Context precision** and **context recall** score **retrieval**: did the right text arrive?
- **Faithfulness** and **answer relevancy** score the **answer**: did the model use the text well?

In [ ]:
def score_samples(samples):
    result = evaluate(EvaluationDataset.from_list(samples), metrics=METRICS, llm=judge,
                      embeddings=embeddings, show_progress=False,
                      run_config=RunConfig(timeout=300, max_workers=4))
    return result.to_pandas()

t0 = time.time()
table_hybrid = score_samples(samples_hybrid)
table_rerank = score_samples(build_samples(rerank=True))
print(f"done in {time.time() - t0:.0f} s")

**You should see:** `done in` some number of seconds, usually a few minutes.

## The result &mdash; the scores, and the weakest stage

A low **retrieval** score means the right text never reached the model: fix chunking, search or
reranking. A low **answer** score with good retrieval means the model had the text and still answered
badly: fix the prompt or the model.

In [ ]:
RETRIEVAL = ["llm_context_precision_with_reference", "context_recall"]
ANSWER = ["faithfulness", "answer_relevancy"]
means = {"hybrid": table_hybrid[RETRIEVAL + ANSWER].mean(),
         "hybrid + rerank": table_rerank[RETRIEVAL + ANSWER].mean()}

print(f"{'metric':40}{'hybrid':>10}{'+ rerank':>10}")
for name in RETRIEVAL + ANSWER:
    print(f"{name:40}{means['hybrid'][name]:>10.2f}{means['hybrid + rerank'][name]:>10.2f}")
for run, m in means.items():
    weakest = "retrieval" if m[RETRIEVAL].mean() < m[ANSWER].mean() else "answer"
    print(f"\n{run:16} weakest stage: {weakest}")

**You should see:** four scores between 0 and 1 for each run. The reranker should raise the
**retrieval** scores most, because it changes which chunks arrive, and the answer scores less.

The judge is a model, so scores move a little between runs. Run Step 4 again and see how much. A
change is real only when it is bigger than that noise. Keep these 6 questions, and score again after
every change to the pipeline.